# **Factorización por algoritmo de Schnorr de un caso simple**

In [35]:
%load_ext autoreload
%autoreload 2

from modules import schnorr_lattice as sl
from modules import qaoa
from modules import teoria_numeros as tn
from modules import utils

import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from itertools import islice
import time
import csv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
seed = 99

In [37]:
N = 48567227
q = 10
c = 4
l = 1.5

In [38]:
cvp = sl.schnorrCVP(N, c, l, seed)

El numero de bits de N = 48567227 es m = 26
La dimension del reticulo que vamos a tratar es n = 8
La cota smooth que vamos a tomar: 128


In [39]:
max_sr_pairs = cvp.smooth_bound + 1
max_iter = 2**cvp.n
elements2sample = 150

In [40]:
i = 0
found = 0
pairs = set()

with open('resolucion_caso_simple_bucle.txt', 'w', newline = '') as f1, \
    open('resolucion_caso_simple_resultado.csv', 'w', newline = '') as f2:

    pair_writer = csv.writer(f2)
    pair_writer.writerow(['u', 'v', 'u - N*v'])

    f1.write(f"Se va factorizar N = {N}; c (parametro de precision) = {c}; l = 1.5; lattice dimension = {cvp.n}; " + 
             f"smooth bound = {cvp.smooth_bound}; numero de pares = {max_sr_pairs}; max_iteraciones = {max_iter}\n")


    while i < max_iter and found < max_sr_pairs: #Itero para conseguir varias instancias del problema

        #Empiezo a medir el tiempo
        start = time.perf_counter_ns()

        instance = cvp.generate_cvp(q, verbose = False)

        cvp_result = cvp.babai_algorithm(instance, delta = 0.75)

        qubo = qaoa.define_qubo(cvp_result.D, cvp_result.res_vector, cvp_result.step_sign, cvp.n)

        Hc, _ = qaoa.define_hamiltonian(qubo)

        circuit = qaoa.construct_circuit(Hc, reps = 1)

        x0 = np.asarray([0.0]*circuit.num_parameters)

        _, optparameters = qaoa.qaoa_algorithm(circuit, Hc, x0)

        results = qaoa.sample_from_parameters(circuit, optparameters, shots = 10_000)

        resultk = dict(islice(results.items(), elements2sample))

        nD = sl.integer_to_matrix(cvp_result.D)

        vnews = sl.bitstring2latticeVectors(nD, resultk.keys(), cvp_result.step_sign, cvp_result.b_op)

        nB = sl.integer_to_matrix(instance.B)

        uv_pairs = tn.vectors2uv_pairs(nB, vnews, cvp.n)

        sr_pairs = tn.uv_pairs2sr_pairs(uv_pairs, cvp.N, cvp.smooth_bound)


        #Fin de medicion del tiempo
        stop = time.perf_counter_ns()
        time_ms = (stop - start) / 1_000_000

        #Calculo de los pares encontrados
        aux = found
        total = len(sr_pairs)

        for par in sr_pairs:
            if par not in pairs:
                pair_writer.writerow([par[0][0], par[0][1], par[1]])
                pairs.add(par)

        found = len(pairs)
        found_iter = found - aux

        f1.write(f"Iteracion {i} encontrado {total} SR Pairs. Encontrados {found_iter} nuevos pares. Se ha tardado {time_ms:.3f}ms\n")

        i = i + 1
    
